# Polygon-Based Genetic Algorithm for Targeted Image Evolution
## Bio-inspired Learning coursework project

Original academic group project by Juan Mallo de la Calle, Sergio Melones Peña and Jesús Rincón Laguarta.

### Section 0. Imports and Dependencies


In [ ]:
import numpy as np
from PIL import Image, ImageDraw, ImageOps
from colour.difference import delta_E_CIE1976
import random, os
from matplotlib import pyplot as plt
from pandas import DataFrame
import pandas as pd
from skimage.metrics import structural_similarity as ssim
from skimage.color import rgb2lab
from skimage import img_as_float32
from scipy.stats import wasserstein_distance
from lpips import LPIPS
import torch
from torchvision.models import inception_v3
from torchvision.models.inception import inception_v3, Inception_V3_Weights
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from PIL import Image

### Section 1. `Individual` Class – One Candidate Solution

**What this block does**

Defines an `Individual`, the basic unit of evolution. Each instance contains:

- An RGBA image (both as a Pillow `Image` and a `numpy` array).
- A `fitness` score initialised to positive infinity.
- Methods that initialise, mutate and evaluate the candidate with respect to a target image.

**Why it is relevant or necessary**

Genetic programming operates on a population of individuals; without this data structure we would have no entities to breed, mutate or evaluate.

**How it fits into the broader process**

Every generation the GP engine creates, mutates and selects among `Individual` objects, gradually improving their resemblance to the target image.

In [ ]:
class Individual:
    '''
    Represents a single candidate image in the population.
    Stores its pixel data, a PIL Image instance and its current fitness.
    '''
    def __init__(self, l, w):
        '''Initialise dimensions, worst-possible fitness and build the first random image'''
        self.l = l
        self.w = w
        self.fitness = float('inf')
        self.array = None
        self.image = None
        self.create_random_image_array()

    def rand_color(self):
        '''Generate a random opaque RGB colour in hexadecimal format'''
        return "#" + ''.join([random.choice('0123456789ABCDEF') for _ in range(6)])

    def create_random_image_array(self):
        '''Build a random RGBA image composed of polygon shapes'''
        iterations = random.randint(8, 12) # Number of polygons to draw
        region = (self.l + self.w) // 8 # Max distance of vertices from polygon centre
        img = Image.new("RGBA", (self.l, self.w), self.rand_color())
        draw = ImageDraw.Draw(img, 'RGBA')

        for _ in range(iterations):
            num_points = random.randint(3, 6) # Triangle -> hexagon
            region_x = random.randint(0, self.l)
            region_y = random.randint(0, self.w)
            # Random vertices around the centre, confined to a square “region”
            xy = [
                (
                    random.randint(region_x - region, region_x + region),
                    random.randint(region_y - region, region_y + region)
                )
                for _ in range(num_points)
            ]
            draw.polygon(xy, fill=self.rand_color())

        self.image = img
        self.array = np.array(img)

    def add_shape(self):
        '''Mutation operator that adds a random polygon to the image'''
        region = random.randint(1, (self.l + self.w) // 4)
        draw = ImageDraw.Draw(self.image, 'RGBA')

        num_points = random.randint(3, 6)
        region_x = random.randint(0, self.l)
        region_y = random.randint(0, self.w)
        xy = [
            (
                random.randint(region_x - region, region_x + region),
                random.randint(region_y - region, region_y + region)
            )
            for _ in range(num_points)
        ]
        draw.polygon(xy, fill=self.rand_color())
        self.array = np.array(self.image)

    def mutate_color_noise(self):
        '''Pixel-level mutation: perturb a random 5 % of the pixels with small colour noise'''
        noisy_array = self.array.copy().astype(np.int16) # int16 avoids overflow
        mask = np.random.rand(*noisy_array.shape[:2]) < 0.05  # 5% pixels
        noise = np.random.randint(-10, 10, size=(self.w, self.l, 4))
        noisy_array[mask] += noise[mask] # Vectorised in-place update
        noisy_array = np.clip(noisy_array, 0, 255).astype(np.uint8)
        self.image = Image.fromarray(noisy_array)
        self.array = noisy_array

    def get_fitness(self, target):
        '''Compute the mean CIE-76 colour distance between this image and the target'''
        self.fitness = np.mean(
            delta_E_CIE1976(
                target.astype('float32'), 
                self.array.astype('float32')
            )
        )
        return self.fitness

### Section 2. `GP` Class – Evolutionary Driver

**What this block does**

Implements the high‑level evolutionary algorithm:
1. Initialises a population of random `Individual`s.
2. Iteratively applies selection crossover, and mutation.
3. Records the best candidate each generation.

**Why it is relevant or necessary**

Encapsulates all evolutionary logic so the main block can run a single method call to evolve images.

**How it fits into the broader process**

Sits atop the `Individual` definition (low‑level genetics) and feeds images to the visualisation helpers.

In [ ]:
class GP:
    '''
    Genetic-Programming engine that evolves Individuals toward a target image
    using selection, crossover and mutation operators.
    '''
    def __init__(self, filename):
        '''Load target image, initialise population and precompute target array'''
        # Load & preprocess the target image
        original = Image.open(filename).convert("RGB")
        original = ImageOps.autocontrast(original) # stretch histogram
        original = original.convert("RGBA")
        original = original.resize((176, 203))
        original.convert("RGB").save("target.png") # keep a reference copy

        # Store target both as PIL image and NumPy array
        self.target_image = original
        self.target_array = np.array(self.target_image).astype(np.float32)

        # Dimensions needed for future Individuals
        self.l, self.w = self.target_image.size

        # Frames for a progress animation (best every 10 generations)
        self.frames = []

        self.saved_structural = False
        self.saved_pixellevel = False
        self.saved_crossover_alpha = False
        self.saved_crossover_mask = False

    def run_gp(self, pop_size=100, epochs=10000):
        """
        Evolve a population of `Individual`s.

        Steps per generation:
        1. Select parents via tournament selection.
        2. Apply a variation operator (two flavours of crossover or mutation).
        3. Keep the offspring only if it improves over its parent(s).
        4. Track the best fitness and periodically save snapshots.
        """
        population = [self.create_individual() for _ in range(pop_size)]

        # Convergence log for later plotting
        data = {'epoch': [], 'fitness_estimate': []}
        snapshot_epochs = np.linspace(0, epochs - 1, 100, dtype=int) # 10 evenly spaced gens
    
        for epoch in range(epochs):
            new_pop = []
            best_fit = float('inf')

            # Generate the next generation
            while len(new_pop) < pop_size:
                p1 = self.tournament(population)
                p2 = self.tournament(population)
                best_fit = min(best_fit, p1.fitness, p2.fitness)

                # Choose operator: 30 % α-blend, 60 % masked crossover, 10 % mutation
                r = random.random()
                if r < 0.3:
                    child = self.crossover(p1, p2)
                elif r < 0.9:
                    child = self.crossover_2(p1, p2)
                else:
                    child = self.mutate(p1)

                # Survivor selection: accept only improving offspring
                if child:
                    new_pop.append(child)
    
            population = new_pop

            # Log progress
            data['epoch'].append(epoch)
            data['fitness_estimate'].append(best_fit)
    
            best = min(population, key=lambda x: x.fitness)

            # Store frame for GIF
            if epoch % 10 == 0:
                self.frames.append(best.image.copy())
                print(f"Gen {epoch} - Fitness: {best.fitness:.2f}")

            # Save PNG snapshots at preset generations
            if epoch in snapshot_epochs:
                best.image.save(f"output/snapshot_gen_{epoch:05d}.png")
    
        # Persist convergence log (pandas.DataFrame expected in scope)
        DataFrame(data).to_csv("data_cross.csv")
        return min(population, key=lambda x: x.fitness)

    def create_individual(self):
        '''Generate a fresh Individual and evaluate its fitness once'''
        ind = Individual(self.l, self.w)
        ind.get_fitness(self.target_array)
        return ind

    def tournament(self, pop, k=6):
        '''Return the fittest of k randomly chosen Individuals'''
        return min(random.sample(pop, k), key=lambda x: x.fitness)

    def crossover(self, ind1, ind2):
        '''
        Uniform α-blending crossover (Image.blend).
        Child survives only if it matches or beats the parents' best fitness
        '''
        if not self.saved_crossover_alpha:
            Image.fromarray(ind1.array).save("crossover_alpha_parent1.png")
            Image.fromarray(ind2.array).save("crossover_alpha_parent2.png")
    
        alpha = random.random()
        child = Individual(self.l, self.w)
        child.image = Image.blend(ind1.image, ind2.image, alpha)
        child.array = np.array(child.image)
        child.get_fitness(self.target_array)

        if not self.saved_crossover_alpha:
            Image.fromarray(child.array).save("crossover_alpha_child.png")
            self.saved_crossover_alpha = True
        
        return child if child.fitness <= min(ind1.fitness, ind2.fitness) else None

    def crossover_2(self, ind1, ind2):
        '''
        Pixel-wise crossover using a random binary mask:
        each mask pixel decides which parent contributes that RGBA value.
        '''
        if not self.saved_crossover_mask:
            Image.fromarray(ind1.array).save("crossover_mask_parent1.png")
            Image.fromarray(ind2.array).save("crossover_mask_parent2.png")
    
        mask = np.random.randint(0, 2, size=(self.w, self.l, 4)) # 0/1 per channel
        img1 = ind1.array * mask
        img2 = ind2.array * (1 - mask)
        combined = (img1 + img2).astype(np.uint8)

        child = Individual(self.l, self.w)
        child.image = Image.fromarray(combined)
        child.array = combined
        child.get_fitness(self.target_array)

        if not self.saved_crossover_mask:
            Image.fromarray(child.array).save("crossover_mask_child.png")
            self.saved_crossover_mask = True
        
        return child if child.fitness <= min(ind1.fitness, ind2.fitness) else None

    def mutate(self, ind):
        '''
        Apply a mutation (shape addition or colour noise) to a clone of *ind*.
        The offspring is kept only if it does not degrade fitness.
        '''
        child = Individual(ind.l, ind.w)
        child.image = ind.image.copy()
        
        if random.random() < 0.5:
            if not self.saved_structural:
                Image.fromarray(np.array(child.image)).save("before_structural_mutation.png")
            child.add_shape() # structural mutation
            if not self.saved_structural:
                Image.fromarray(np.array(child.image)).save("after_structural_mutation.png")
                self.saved_structural = True
            
        else:
            if not self.saved_pixellevel:
                Image.fromarray(np.array(child.image)).save("before_pixellevel_mutation.png")
            child.array = np.array(child.image)
            child.mutate_color_noise() # pixel-level mutation
            if not self.saved_pixellevel:
                Image.fromarray(np.array(child.image)).save("after_pixellevel_mutation.png")
                self.saved_pixellevel = True
            
        child.get_fitness(self.target_array)
        return child if child.fitness <= ind.fitness else None

### Section 3. `generate_gif` Utility Function

**What this block does**

Takes the list of best images saved during evolution and stitches them into an animated GIF so we can watch the convergence process.

**Why it is relevant or necessary**

Provides an intuitive, shareable visual summary of the algorithm’s progress.

**How it fits into the broader process**

Runs after the GP cycle completes, using the frames collected inside the `GP` instance.

In [ ]:
def generate_gif(frames, gif_name="evolution1.gif", duration=200):
    '''Save the sequence of best images as an animated GIF'''
    if not frames:
        print("No frames found.")
        return
        
    frames[0].save(gif_name, save_all=True, append_images=frames[1:], duration=duration, loop=0)
    print(f"GIF saved as {gif_name}")

### Section 4. Main Execution – Bringing It All Together

**What this block does**

Creates an output folder, configures and launches the genetic program for 100 generations x 10000 evaluation budget, shows the best evolved image, and exports an `.gif` of the entire run.

**Why it is relevant or necessary**

Shows how to use the previously defined classes and functions in practice.

**How it fits into the broader process**

Acts as the entry point: initialise, evolve, visualise, and persist results.

In [ ]:
if __name__ == "__main__":
    # Execute the evolutionary process
    os.makedirs("output", exist_ok=True)
    gp = GP("target.png")
    best = gp.run_gp(120, 20000)
    plt.imshow(best.image)
    plt.axis('off')
    plt.title("Best Individual")
    plt.show()
    generate_gif(gp.frames)

### Section 5. Evaluation – Objective Metrics and Visualizations

**What this block does**

This section defines and executes a comprehensive evaluation of the best image generated by the genetic art algorithm. It computes a series of objective quality metrics to assess similarity between the generated image and the original target image. The evaluated metrics include:

- MSE (Mean Squared Error): Measures average pixel-wise intensity differences.
- PSNR (Peak Signal-to-Noise Ratio): Indicates the signal quality; higher is better.
- SSIM (Structural Similarity Index): Captures perceptual image similarity in terms of structure and luminance.
- ΔE\* (CIE-76): A perceptual metric measuring color difference in the Lab color space.
- Histogram Distance (Wasserstein): Quantifies distributional differences across color channels.
- LPIPS: A deep perceptual metric based on pretrained neural features (AlexNet).
- FID (simplified version): Measures distance between high-level features extracted from a pretrained InceptionV3 model.

It also generates three visual outputs:
1. The original target image.
2. The final generated image.
3. A heatmap of ΔE\* values highlighting local color differences.

Finally, the `plot_learning_curve()` function visualizes the convergence behavior of the evolutionary algorithm using the fitness scores logged during training.

**Why it is relevant or necessary**

Objective metrics and visualizations provide rigorous, interpretable evidence of the quality and convergence of the evolutionary image generation process. Metrics such as SSIM, ΔE\*, and LPIPS correlate better with human perception than raw pixel-wise error. Including multiple complementary metrics allows for a more nuanced evaluation.

Visualizations (like the ΔE\* heatmap) help identify *where* the generated image deviates most from the target, which is useful for debugging, refinement, and guiding future improvements in the evolutionary operators.

**How it fits into the broader process**

This section serves as the final diagnostic step after the evolutionary loop (`run_gp`) has been executed. It verifies whether the generated output has converged towards the intended target, and how well. The evaluation informs both qualitative and quantitative assessment of the system's performance, and it provides empirical support for conclusions drawn in analysis or reporting.

In [ ]:
# Function to compute Fréchet Inception Distance (FID) between two images
def compute_fid(img1, img2, device='cpu'):
    '''
    Approximate FID for two single RGB images using InceptionV3 features.
    If only one image per domain is used, skips covariance terms.
    '''
    # Load model
    model = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, transform_input=False)
    model.fc = torch.nn.Identity()
    model.eval().to(device)

    preprocess = Compose([
        Resize((299, 299)),
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225])
    ])

    def get_features(image):
        x = preprocess(image.convert("RGB")).unsqueeze(0).to(device)
        with torch.no_grad():
            features = model(x)
        return features.cpu().numpy().squeeze()

    # Extract features
    act1 = get_features(img1)
    act2 = get_features(img2)

    # Treat them as means (no covariance)
    diff = act1 - act2
    fid = np.sum(diff ** 2)
    return float(fid)

def evaluate_result(best_individual, gp_engine):
    '''
    Evaluates the generated image against the target image using:
    - MSE
    - PSNR
    - SSIM
    - ΔE* (CIE-76)
    - Histogram distance (Wasserstein)
    - LPIPS (deep perceptual similarity)
    - FID (Fréchet Inception Distance)
    '''
    # RGB data
    target_rgb = gp_engine.target_array[..., :3].astype(np.uint8)
    generated_rgb = best_individual.array[..., :3].astype(np.uint8)

    # Float for perceptual metrics
    target_f = img_as_float32(target_rgb)
    generated_f = img_as_float32(generated_rgb)

    # Pixel-wise metrics
    mse = np.mean((target_rgb.astype(np.float32) - generated_rgb.astype(np.float32)) ** 2)
    psnr = 10 * np.log10((255 ** 2) / mse)
    ssim_val = ssim(target_f, generated_f, channel_axis=-1, data_range=1.0)

    # ΔE*
    target_lab = rgb2lab(target_f)
    generated_lab = rgb2lab(generated_f)
    delta_e = np.linalg.norm(target_lab - generated_lab, axis=-1)
    mean_delta_e = delta_e.mean()

    # Histogram difference (Wasserstein distance per channel)
    hist_diff = np.mean([
        wasserstein_distance(target_rgb[..., i].flatten(), generated_rgb[..., i].flatten())
        for i in range(3)
    ])

    # LPIPS
    loss_fn = LPIPS(net='alex')
    img1_t = torch.tensor(target_f).permute(2, 0, 1).unsqueeze(0) * 2 - 1
    img2_t = torch.tensor(generated_f).permute(2, 0, 1).unsqueeze(0) * 2 - 1
    lpips_score = loss_fn(img1_t, img2_t).item()

    # FID
    fid_score = compute_fid(Image.fromarray(target_rgb), Image.fromarray(generated_rgb))

    # Print results
    print(f"MSE        : {mse:.2f}")
    print(f"PSNR       : {psnr:.2f} dB")
    print(f"SSIM       : {ssim_val:.4f}")
    print(f"ΔE* (Lab)  : {mean_delta_e:.2f}")
    print(f"Histogram  : {hist_diff:.2f}")
    print(f"LPIPS      : {lpips_score:.4f}")
    print(f"FID        : {fid_score:.2f}")

    # 3. Visualizations
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(target_rgb)
    ax[0].set_title("Target Image")
    ax[0].axis("off")

    ax[1].imshow(generated_rgb)
    ax[1].set_title("Generated Image")
    ax[1].axis("off")

    im = ax[2].imshow(delta_e, cmap="inferno", vmin=0, vmax=np.percentile(delta_e, 99))
    ax[2].set_title("ΔE* Heatmap (CIE-76)")
    ax[2].axis("off")
    plt.colorbar(im, ax=ax[2], fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

# Learning curve plot
def plot_learning_curve(csv_path="data_cross.csv"):
    '''
    Plots the fitness evolution over generations from CSV log.
    '''
    data = pd.read_csv(csv_path)
    plt.figure(figsize=(8, 5))
    plt.plot(data["epoch"], data["fitness_estimate"], label="Best Fitness")
    plt.xlabel("Generation")
    plt.ylabel("Fitness (CIE ΔE*)")
    plt.title("Fitness over Generations")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
evaluate_result(best, gp)

In [ ]:
plot_learning_curve(csv_path="data_cross.csv")